In [1]:
!pip uninstall -y langchain langchain-core langchain-community langchain-text-splitters langchain-groq

Found existing installation: langchain 1.3.2
Uninstalling langchain-1.3.2:
  Successfully uninstalled langchain-1.3.2
Found existing installation: langchain-core 1.4.0
Uninstalling langchain-core-1.4.0:
  Successfully uninstalled langchain-core-1.4.0
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
Found existing installation: langchain-text-splitters 1.1.2
Uninstalling langchain-text-splitters-1.1.2:
  Successfully uninstalled langchain-text-splitters-1.1.2
Found existing installation: langchain-groq 1.1.2
Uninstalling langchain-groq-1.1.2:
  Successfully uninstalled langchain-groq-1.1.2


In [2]:
!pip install -U langchain
!pip install -U langchain-community
!pip install -U langchain-groq
!pip install -U faiss-cpu sentence-transformers pypdf

  Using cached langchain-1.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_core-1.4.0-py3-none-any.whl.metadata (4.5 kB)
Using cached langchain-1.3.2-py3-none-any.whl (121 kB)
Using cached langchain_core-1.4.0-py3-none-any.whl (548 kB)

   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   -------------------- ------------------- 1/2 [langchain]
   ---------------------------------------- 2/2 [langchain]



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.7 requires langchain-text-splitters<2.0.0,>=1.1.2, which is not installed.


  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)
Using cached langchain_text_splitters-1.1.2-py3-none-any.whl (35 kB)

   -------------------- ------------------- 1/2 [langchain-community]
   -------------------- ------------------- 1/2 [langchain-community]
   -------------------- ------------------- 1/2 [langchain-community]
   -------------------- ------------------- 1/2 [langchain-community]
   -------------------- ------------------- 1/2 [langchain-community]
   -------------------- ------------------- 1/2 [langchain-community]
   -------------------- ------------------- 1/2 [langchain-community]
   -------------------- ------------------- 1/2 [langchain-community]
   -------------------- ------------------- 1/2 [langchain-community]
   -------------------- ------------------- 1/2 [langchain-community]
   -----

In [3]:
!pip install -U langchain-text-splitters

In [9]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from groq import Groq
pdf_path = r"C:\Users\hp\Downloads\DeepLearning_Notes.pdf"  
loader = PyPDFLoader(pdf_path)
documents = loader.load()
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks = splitter.split_documents(documents)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever()
print("FAISS index created successfully!")
client = Groq(api_key="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")
def answer_question(question):
    # retrieve relevant chunks
    docs = retriever.invoke(question)
    context = "\n".join([doc.page_content for doc in docs])
    prompt = f"""
    You are a helpful assistant.
    Answer the question using only the context below.
    Context:
    {context}
    Question:
    {question}
    """
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content
questions = [
    "What is the main topic of the document?",
    "Explain the key concepts mentioned.",
    "Summarize the document in simple terms.",
    "What are the important points to remember?",
    "Give a short conclusion of the PDF."
]
print("\n===== RAG Q&A RESULTS =====\n")
for i, q in enumerate(questions, 1):
    ans = answer_question(q)
    print(f"Q{i}: {q}")
    print(f"A{i}: {ans}")
    print("-" * 80)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS index created successfully!

===== RAG Q&A RESULTS =====

Q1: What is the main topic of the document?
A1: The main topic of the document is "Deep Learning Foundations & Generative AI" specifically, Week 3 of an AI/ML internship program.
--------------------------------------------------------------------------------
Q2: Explain the key concepts mentioned.
A2: The key concepts mentioned are:

1. **Neural Networks**: A system of connected layers that learn different levels of abstraction from data, loosely inspired by how neurons in the brain connect and signal each other.

2. **Layers**: Every neural network has three types of layers:
   - **Input Layer**: Receives the raw features from the data.
   - **Hidden Layer(s)**: Learns patterns from the data and can have multiple layers, making it 'deep'.
   - **Output Layer**: Produces the final answer or prediction.

3. **Activation Functions**: Give neural networks the power to learn complex, non-linear patterns.

4. **Weights & Biase